In [1]:
import os
%pwd

'd:\\Data Science\\END to END Proj\\Introvert vs Extrovert\\Introvert-Vs-Extrovert\\research'

In [2]:
os.chdir("../")

In [3]:
import dagshub
dagshub.init(repo_owner='gowtham-dd', repo_name='Introvert-Vs-Extrovert', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as gowtham-dd

Initialized MLflow to track repo "gowtham-dd/Introvert-Vs-Extrovert"

Repository gowtham-dd/Introvert-Vs-Extrovert initialized!

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    x_val_path: Path
    y_val_path: Path
    model_dir: Path
    ensemble_path: Path
    ordinal_encoder_path: Path
    label_encoder_path: Path
    metric_file: Path
    mlflow_uri: str
    ensemble_weights: dict           # from params.yaml


In [5]:
from src.IntrovertVsExtrovert.utils.common import read_yaml, create_directories,save_json
from src.IntrovertVsExtrovert.constant import *

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        cfg   = self.config.model_evaluation
        train = self.params.training        # has ensemble_weights
        create_directories([cfg.root_dir])

        return ModelEvaluationConfig(
            root_dir             = Path(cfg.root_dir),
            x_val_path           = Path(cfg.x_val_path),
            y_val_path           = Path(cfg.y_val_path),
            model_dir            = Path(cfg.model_dir),
            ensemble_path        = Path(cfg.ensemble_path),
            ordinal_encoder_path = Path(cfg.ordinal_encoder_path),
            label_encoder_path   = Path(cfg.label_encoder_path),
            metric_file          = Path(cfg.metric_file),
            mlflow_uri           = cfg.mlflow_uri,
            ensemble_weights     = dict(train.ensemble_weights)
        )


In [6]:
import json, joblib, os, warnings, numpy as np, pandas as pd
from pathlib import Path
from typing import List, Dict
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import (log_loss, accuracy_score, roc_auc_score,
                             f1_score, precision_score, recall_score)
import mlflow
from urllib.parse import urlparse

warnings.filterwarnings("ignore")


class ModelEvaluation:
    def __init__(self, cfg: ModelEvaluationConfig):
        self.cfg = cfg

    # ------------------------------------------------------------------
    def _load_xy(self):
        X = pd.read_parquet(self.cfg.x_val_path)
        y = pd.read_csv(self.cfg.y_val_path, header=None).squeeze("columns")
        return X, y

    # ------------------------------------------------------------------
    def _load_xgb_models(self) -> List[xgb.Booster]:
        models = []
        for p in sorted(self.cfg.model_dir.glob("xgb_fold*.bin")):
            model = xgb.Booster()
            model.load_model(str(p))
            models.append(model)
        if not models:
            raise FileNotFoundError("No XGB models found in model_dir")
        return models

    def _load_cat_models(self) -> List[CatBoostClassifier]:
        models = []
        for p in sorted(self.cfg.model_dir.glob("cat_fold*.cbm")):
            m = CatBoostClassifier()
            m.load_model(str(p))
            models.append(m)
        if not models:
            raise FileNotFoundError("No CatBoost models found in model_dir")
        return models

    # ------------------------------------------------------------------
    def _average_preds(self, preds: List[np.ndarray]) -> np.ndarray:
        return np.mean(np.vstack(preds), axis=0)

    # ------------------------------------------------------------------
    def evaluate(self) -> Dict[str, float]:
        # 1 Load data
        X, y = self._load_xy()

        # 2 Load models & get probabilities
        xgb_probs = self._average_preds([
            m.predict(xgb.DMatrix(X)) for m in self._load_xgb_models()
        ])
        cat_probs = self._average_preds([
            m.predict_proba(X)[:, 1] for m in self._load_cat_models()
        ])

        # 3 Blend
        if Path(self.cfg.ensemble_path).exists():
            weights = joblib.load(self.cfg.ensemble_path)
        else:
            weights = self.cfg.ensemble_weights   # from params.yaml

        blended = weights["xgb"] * xgb_probs + weights["cat"] * cat_probs
        preds   = (blended >= 0.5).astype(int)

        # 4 Metrics
        metrics = {
            "accuracy":  accuracy_score(y, preds),
            "log_loss":  log_loss(y, blended),
            "f1_score":  f1_score(y, preds),
            "precision": precision_score(y, preds),
            "recall":    recall_score(y, preds),
            "roc_auc":   roc_auc_score(y, blended),
        }

        # 5 Persist metrics as JSON
        save_json(self.cfg.metric_file, metrics)
        print(f"Metrics saved ➜ {self.cfg.metric_file}")
        return metrics

    # ------------------------------------------------------------------
    def log_mlflow(self):
        metrics = self.evaluate()

        mlflow.set_tracking_uri(self.cfg.mlflow_uri)
        scheme = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run(run_name="model-eval"):
            # params: just ensemble weights here
            mlflow.log_params(self.cfg.ensemble_weights)

            for k, v in metrics.items():
                mlflow.log_metric(k, float(v))

            # log artifacts
            mlflow.log_artifact(str(self.cfg.metric_file), artifact_path="metrics")
            mlflow.log_artifact(str(self.cfg.ordinal_encoder_path), artifact_path="encoders")
            mlflow.log_artifact(str(self.cfg.label_encoder_path),   artifact_path="encoders")

            # register first model of each type (optional)
            if scheme != "file":
                xgb0 = self._load_xgb_models()[0]
                mlflow.xgboost.log_model(
                    xgb_model=xgb0,
                    artifact_path="xgb_model",
                    registered_model_name="IntroExtro_XGB"
                )

                cat0 = self._load_cat_models()[0]
                mlflow.catboost.log_model(
                    cb_model=cat0,
                    artifact_path="cat_model",
                    registered_model_name="IntroExtro_Cat"
                )


            print("✅ Metrics & artifacts logged to MLflow.")



In [7]:
try:
        config_manager = ConfigurationManager()
        eval_config = config_manager.get_model_evaluation_config()

        evaluator = ModelEvaluation(eval_config)
        evaluator.log_mlflow()

except Exception as e:
        print(f"❌ Exception during model‑evaluation stage: {e}")

[2025-07-12 14:11:36,039: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-12 14:11:36,052: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-12 14:11:36,060: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-07-12 14:11:36,066: INFO: common: created directory at: artifacts]
[2025-07-12 14:11:36,069: INFO: common: created directory at: artifacts/model_evaluation]
[2025-07-12 14:11:36,474: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]
Metrics saved ➜ artifacts\model_evaluation\metrics.json


Registered model 'IntroExtro_XGB' already exists. Creating a new version of this model...
2025/07/12 14:12:03 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: IntroExtro_XGB, version 2
Created version '2' of model 'IntroExtro_XGB'.
Successfully registered model 'IntroExtro_Cat'.
2025/07/12 14:12:18 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: IntroExtro_Cat, version 1
Created version '1' of model 'IntroExtro_Cat'.


✅ Metrics & artifacts logged to MLflow.
